# Lab 04 · Part 2 — Full fine-tuning, LoRA and QLoRA

Part 1 left the weights alone. Here the weights are trained on labelled
tickets, three ways:

| method | what is trained | base weights stored as |
|---|---|---|
| **full fine-tuning** | every transformer weight (~106M) | fp32 |
| **LoRA** | two small matrices per layer (~0.9M) | fp32, frozen |
| **QLoRA** | the same LoRA matrices | 4-bit NF4, frozen |

LoRA and the NF4 quantizer are written from scratch (`lora.py`, `quant.py`),
no `peft` or `bitsandbytes`. `tests/test_methods.py` checks them.

All three train on the plain zero-shot prompt, so a fine-tuned model needs no
examples at inference time (unlike few-shot).

In [1]:
import sys, json
from pathlib import Path
sys.path.insert(0, ".")

import numpy as np
import pandas as pd
import torch

import finetune, lora, quant
from tickets import LABELS, load_tickets, load_general_sentences, zero_shot_prompt
from model_utils import (load_model, label_token_ids, evaluate, general_perplexity, count_params)

RESULTS = Path("results")
RESULTS.mkdir(exist_ok=True)
train_df, test_df = load_tickets()
general = load_general_sentences()
test_prompts = [zero_shot_prompt(t) for t in test_df["text"]]
test_labels = test_df["label"].tolist()
prompting = json.load(open(RESULTS / "prompting.json"))   # written by notebook 1

## LoRA in one layer

A frozen weight `W` (out × in) gets a trainable low-rank update:

`W' = W + (alpha / r) · B A`, with `A` of shape `r × in` and `B` of shape `out × r`.

`B` starts at zero, so training starts from exactly the pretrained model. For a
`576 × 576` attention projection and rank `r = 8` that is `8 × (576 + 576) = 9,216`
trainable numbers instead of `331,776`.

In [2]:
m, tok = finetune.build_lora(r=8, alpha=16)
layer = m.model.layers[0].self_attn.q_proj
print(layer)
print()
trainable, total = count_params(m)
print(f"wrapped layers : {len(lora.lora_layers(m))}")
print(f"trainable      : {trainable:,}")
print(f"total          : {total:,}")
print(f"trainable share: {100 * trainable / total:.2f}%")
del m

LoRALinear(
  (base): Linear(in_features=576, out_features=576, bias=False)
  (dropout): Dropout(p=0.05, inplace=False)
)

wrapped layers : 120
trainable      : 921,600
total          : 135,436,608
trainable share: 0.68%


## QLoRA: what does quantizing cost?

Each weight matrix is cut into blocks of 64 values. Every block is divided by
its largest absolute value, so it lands in [-1, 1], and each value is replaced
by the nearest of 16 NF4 levels (4 bits, two packed per byte). Blocks keep one
fp32 scale each. In `forward()` the weights are expanded back to fp32.

First the error on a single real weight matrix, then what it does to the whole
model *before any training*.

In [3]:
base, tok = load_model()
w = base.model.layers[0].mlp.gate_proj.weight
print("gate_proj of layer 0:", tuple(w.shape), f"std {w.std():.4f}")
for bits in (8, 4):
    q = quant.QuantLinear(base.model.layers[0].mlp.gate_proj, bits=bits)
    err = (q.weight_fp32() - w).abs().mean().item()
    print(f"  {bits}-bit: mean abs error {err:.5f}   "
          f"{quant.tensor_bytes(q) / 1e3:.0f} KB vs {w.numel() * 4 / 1e3:.0f} KB in fp32")
del base

gate_proj of layer 0: (1536, 576) std 0.1844
  8-bit: mean abs error 0.00094   940 KB vs 3539 KB in fp32
  4-bit: mean abs error 0.01350   498 KB vs 3539 KB in fp32


In [4]:
label_ids = label_token_ids(tok)
quant_rows = [{"setting": "fp32 (original)", "accuracy": prompting["zero_shot"]["accuracy"],
               "general_ppl": prompting["base_general_ppl"], "decoder_MB": None}]
for bits in (8, 4):
    m, _ = finetune.build_qlora(bits=bits)   # LoRA B = 0, so this IS the quantized base model
    r = evaluate(m, tok, test_prompts, test_labels, label_ids)
    quant_rows.append({"setting": f"{bits}-bit quantized", "accuracy": r["accuracy"],
                       "general_ppl": general_perplexity(m, tok, general),
                       "decoder_MB": quant.tensor_bytes(m.model.layers) / 1e6})
    del m
m, _ = load_model()
quant_rows[0]["decoder_MB"] = quant.tensor_bytes(m.model.layers) / 1e6
del m
quant_df = pd.DataFrame(quant_rows).round(3)
json.dump(quant_rows, open(RESULTS / "quantization.json", "w"), indent=1)
quant_df

,setting,accuracy,general_ppl,decoder_MB
0,fp32 (original),0.44,32.274,424.812
1,8-bit quantized,0.45,32.346,116.628
2,4-bit quantized,0.23,38.910,63.544


The table is the zero-shot model (no training anywhere) at three precisions.
`decoder_MB` counts the 30 transformer blocks only, not the embedding. The two quantized rows also include the 3.7 MB of (untrained) LoRA matrices that `build_qlora` adds, so their weights are ~3.7 MB smaller than shown.

8-bit costs nothing visible (45% vs 44%, perplexity 32.3 vs 32.3) at 27% of the size. 4-bit is where it hurts: accuracy falls to 23% and perplexity rises to 38.9. A 135M model has little redundancy to spare.

## Training runs

Every run: 5 epochs, batch size 8, AdamW, test on the 100 held-out tickets.
Learning rates are the usual defaults for each style (1e-5 for full fine-tuning,
1e-3 for LoRA adapters); I did not tune them per method.
Each run's result is appended to `results/finetune.json` right away, so if the
kernel dies you can re-run the notebook and it picks up where it stopped.

### A. Main comparison: 20 labelled tickets per category (100 total), 3 seeds

In [5]:
FT_PATH = RESULTS / "finetune.json"
runs = json.load(open(FT_PATH)) if FT_PATH.exists() else []

def get_or_run(method, n, seed, r=8, bits=4, epochs=5):
    key = (method, n, seed, r, bits if method == "qlora" else None, epochs)
    for run in runs:
        if (run["method"], run["n_per_class"], run["seed"], run["r"], run.get("bits"), run["epochs"]) == key:
            return run
    res = finetune.run_experiment(method, train_df, test_df, general, n, seed, r=r, epochs=epochs, bits=bits)
    runs.append(res)
    json.dump(runs, open(FT_PATH, "w"))
    print(f"{method:5s} n={n:2d} r={r:2d} seed={seed}  acc {res['accuracy']:.3f}  "
          f"F1 {res['macro_f1']:.3f}  ppl {res['general_ppl']:.1f}  train {res['train_sec']:.0f}s")
    return res

for seed in [0, 1, 2]:
    for method in ["full", "lora", "qlora"]:
        get_or_run(method, 20, seed)

full  n=20 r= 8 seed=0  acc 0.900  F1 0.900  ppl 32.5  train 15s
lora  n=20 r= 8 seed=0  acc 0.880  F1 0.879  ppl 32.9  train 17s
qlora n=20 r= 8 seed=0  acc 0.920  F1 0.920  ppl 39.1  train 23s
full  n=20 r= 8 seed=1  acc 0.900  F1 0.901  ppl 32.6  train 16s
lora  n=20 r= 8 seed=1  acc 0.890  F1 0.892  ppl 33.7  train 18s
qlora n=20 r= 8 seed=1  acc 0.940  F1 0.939  ppl 38.8  train 23s
full  n=20 r= 8 seed=2  acc 0.910  F1 0.910  ppl 32.6  train 14s
lora  n=20 r= 8 seed=2  acc 0.930  F1 0.929  ppl 32.9  train 12s
qlora n=20 r= 8 seed=2  acc 0.950  F1 0.950  ppl 38.8  train 16s


In [6]:
main = pd.DataFrame([r for r in runs if r["n_per_class"] == 20 and r["r"] == 8 and r["epochs"] == 5
                     and r.get("bits", 4) == 4])
main.groupby("method").agg(acc=("accuracy", "mean"), acc_std=("accuracy", "std"),
                           f1=("macro_f1", "mean"), general_ppl=("general_ppl", "mean"),
                           trainable=("trainable_params", "first"),
                           train_sec=("train_sec", "mean")).round(3)

,acc,acc_std,f1,general_ppl,trainable,train_sec
method,,,,,,
full,0.903,0.006,0.904,32.554,106203456,14.749
lora,0.900,0.026,0.900,33.176,921600,15.739
qlora,0.937,0.015,0.936,38.898,921600,20.664


### B. How many labelled tickets does LoRA need?

Same LoRA setup, 1, 2, 5, 10 and 20 tickets per category (n=20 is from A).
This is the fine-tuning counterpart of Part 1's few-shot `k`.

In [7]:
for n in [1, 2, 5, 10]:
    for seed in [0, 1, 2]:
        get_or_run("lora", n, seed)

lora_n = pd.DataFrame([r for r in runs if r["method"] == "lora" and r["r"] == 8 and r["epochs"] == 5])
lora_n.groupby("n_per_class").agg(acc=("accuracy", "mean"), acc_std=("accuracy", "std"),
                                  f1=("macro_f1", "mean")).round(3)

lora  n= 1 r= 8 seed=0  acc 0.870  F1 0.870  ppl 32.6  train 1s
lora  n= 1 r= 8 seed=1  acc 0.860  F1 0.860  ppl 32.7  train 1s
lora  n= 1 r= 8 seed=2  acc 0.760  F1 0.745  ppl 32.5  train 1s
lora  n= 2 r= 8 seed=0  acc 0.870  F1 0.870  ppl 32.9  train 2s
lora  n= 2 r= 8 seed=1  acc 0.850  F1 0.847  ppl 32.6  train 2s
lora  n= 2 r= 8 seed=2  acc 0.850  F1 0.848  ppl 32.7  train 2s
lora  n= 5 r= 8 seed=0  acc 0.850  F1 0.847  ppl 32.9  train 4s
lora  n= 5 r= 8 seed=1  acc 0.850  F1 0.850  ppl 32.7  train 4s
lora  n= 5 r= 8 seed=2  acc 0.810  F1 0.812  ppl 32.5  train 4s
lora  n=10 r= 8 seed=0  acc 0.890  F1 0.892  ppl 32.8  train 7s
lora  n=10 r= 8 seed=1  acc 0.880  F1 0.880  ppl 33.4  train 7s
lora  n=10 r= 8 seed=2  acc 0.870  F1 0.872  ppl 33.4  train 6s


,acc,acc_std,f1
n_per_class,,,
1,0.830,0.061,0.825
2,0.857,0.012,0.855
5,0.837,0.023,0.837
10,0.880,0.010,0.881
20,0.900,0.026,0.900


### C. Does the rank matter?

LoRA rank `r` at 10 tickets per category, seeds 0 and 1, with `alpha = 2r`
(so the update's scale stays the same).

In [8]:
for r in [1, 4, 8, 16, 32]:
    for seed in [0, 1]:
        get_or_run("lora", 10, seed, r=r)

rank_df = pd.DataFrame([x for x in runs if x["method"] == "lora" and x["n_per_class"] == 10
                        and x["seed"] in (0, 1) and x["epochs"] == 5])
rank_df.groupby("r").agg(acc=("accuracy", "mean"), trainable=("trainable_params", "first"),
                         checkpoint_KB=("checkpoint_bytes", lambda s: s.iloc[0] / 1e3)).round(3)

lora  n=10 r= 1 seed=0  acc 0.860  F1 0.860  ppl 32.5  train 6s
lora  n=10 r= 1 seed=1  acc 0.860  F1 0.860  ppl 32.6  train 6s
lora  n=10 r= 4 seed=0  acc 0.860  F1 0.862  ppl 32.6  train 6s
lora  n=10 r= 4 seed=1  acc 0.870  F1 0.870  ppl 33.2  train 6s
lora  n=10 r=16 seed=0  acc 0.850  F1 0.851  ppl 33.9  train 7s
lora  n=10 r=16 seed=1  acc 0.880  F1 0.880  ppl 33.3  train 7s
lora  n=10 r=32 seed=0  acc 0.840  F1 0.839  ppl 35.6  train 7s
lora  n=10 r=32 seed=1  acc 0.890  F1 0.890  ppl 33.8  train 7s


,acc,trainable,checkpoint_KB
r,,,
1,0.860,115200,537.689
4,0.865,460800,1920.153
8,0.885,921600,3763.353
16,0.865,1843200,7449.999
32,0.865,3686400,14822.799


## The adapter is a separate, switchable file

Because the base weights never changed, a LoRA fine-tune is just a small file.
Three things to check on the main LoRA run (n=20, seed 0):

1. loading the saved adapter into a fresh base model reproduces the run
2. switching the adapter off gives back the zero-shot model exactly
3. merging the adapter into the weights keeps the predictions (and removes the extra compute)

In [9]:
ref = [r for r in runs if r["method"] == "lora" and r["n_per_class"] == 20 and r["seed"] == 0
       and r["r"] == 8 and r["epochs"] == 5][0]
adapter_path = finetune.ARTIFACT_DIR / "lora_r8_n20_s0.pt"
if not adapter_path.exists():           # artifacts/ is not committed
    ref = finetune.run_experiment("lora", train_df, test_df, general, 20, 0)
print(f"adapter file: {adapter_path.stat().st_size / 1e6:.2f} MB")

m, tok = finetune.build_lora(r=8, alpha=16)
lora.load_lora_state_dict(m, torch.load(adapter_path))
a = evaluate(m, tok, test_prompts, test_labels, label_ids)
print("1. loaded adapter reproduces the trained run :", a["preds"] == ref["preds"], f"(acc {a['accuracy']:.2f})")

lora.set_lora_enabled(m, False)
b = evaluate(m, tok, test_prompts, test_labels, label_ids)
print("2. adapter off == zero-shot model            :", b["preds"] == prompting["zero_shot"]["preds"],
      f"(acc {b['accuracy']:.2f})")

lora.set_lora_enabled(m, True)
lora.merge_lora(m)
c = evaluate(m, tok, test_prompts, test_labels, label_ids)
agree = np.mean(np.array(c["preds"]) == np.array(a["preds"]))
print(f"3. merged model agrees with adapter model    : {agree:.0%} of predictions")
print(f"   seconds per ticket: with adapter {a['sec_per_ticket']:.3f}  merged {c['sec_per_ticket']:.3f}")
del m

adapter file: 3.76 MB
1. loaded adapter reproduces the trained run : True (acc 0.88)
2. adapter off == zero-shot model            : True (acc 0.44)
3. merged model agrees with adapter model    : 100% of predictions
   seconds per ticket: with adapter 0.008  merged 0.006


## What I saw

* **Fine-tuning works, and cheaply.** With 20 tickets per category all three methods reach
  ~90% or better, against 44% zero-shot and 45% for 4-shot prompting. LoRA trains 0.68% of the weights.
* **Full fine-tuning vs LoRA is a tie at this size.** 90.3% ± 0.6 vs 90.0% ± 2.6 (3 seeds).
  LoRA's adapter file is 3.8 MB; full fine-tuning has to ship the 425 MB of weights it changed.
* **QLoRA came out highest (93.7%) even though the 4-bit base started at 23%.**
  The adapters repaired the damage from quantization completely. I did not expect QLoRA to beat
  LoRA, and with 3 seeds and a 100-ticket test set (1 ticket = 1 point) I would not read much into
  the 3-4 point gap. I did not test why.
* **The perplexity jump in QLoRA is the quantization, not the training.** General-text perplexity
  is 38.9 after training, the same as the 4-bit model before any training (38.9). LoRA and full fine-tuning
  moved it by under one point (32.3 to 32.6-33.2), so at this scale there was no real catastrophic forgetting.
* **Few labelled tickets go a long way.** One ticket per category (5 total) already gives 83%,
  and 20 per category gives 90%. The curve is quite flat after n=2.
* **Rank did not matter here.** r = 1 to 32 all land in 84-89% at n=10, within seed noise.
  I'd expect rank to matter on a harder task.
* **The adapter really is separable.** Reloading the saved file reproduces the run exactly; switching it off
  gives back the zero-shot predictions exactly; merging it changed no predictions.